# Lab 01 — Linear Regression — Salary vs Experience
**Regression Track** · Beginner · ~40 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Fit simple OLS with sklearn LinearRegression
2. Interpret slope and intercept in business terms
3. Evaluate with R² and residual plots
4. Predict salary at unseen experience values

## Datasets (this folder)
- `Salary_Data.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-01-linear-regression-salary/lab-01-linear-regression-salary.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Salary_Data.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-01-linear-regression-salary"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-01-linear-regression-salary/bundle/dataset.zip"
NEED = ["Salary_Data.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Data Science: Predicting Salary from Experience

> **Scenario:** You are an HR analyst at a mid-size tech company. Leadership wants a defensible model that predicts expected salary from years of experience, so they can benchmark offers and spot pay inequities. You have 30 historical employee records with `YearsExperience` and `Salary`. Your job is to build, evaluate, and interpret a simple linear regression model.
>
> **You will learn:** OLS intuition, sklearn LinearRegression, slope/intercept interpretation, R², residual analysis, train/test split.
> **Time:** ~40 minutes. **Level:** Beginner. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | What it means | Where it shows up |
|---|---|---|
| Linear regression | Fit a straight line `y = mx + b` minimizing squared error | `LinearRegression().fit(X, y)` |
| Slope (m) | Change in salary per extra year of experience | `model.coef_` |
| Intercept (b) | Predicted salary at 0 years experience | `model.intercept_` |
| R² | Fraction of variance explained (1 = perfect) | `model.score(X_test, y_test)` |
| Residuals | Prediction errors (y_true − y_pred) | Residual plot |
| Train/test split | Hold out data to estimate real-world performance | `train_test_split` |

---

## 1. Load and Explore the Data

Every data scientist's first model is a straight line — and for good reason. Linear regression is the "hello world" of predictive modelling: simple enough to fit in one line of code, yet powerful enough to answer real business questions. When an HR team asks "what should we offer a candidate with 5 years of experience?", they are asking for a prediction. When they ask "are we paying fairly across experience levels?", they are asking for a relationship. Linear regression answers both.

The idea is beautifully simple. Imagine plotting every employee on a graph: years of experience on the x-axis, salary on the y-axis. Each person is a dot. Now draw a straight line through the cloud of dots that minimises the total squared distance from each dot to the line. That line *is* your model. The slope tells you how much salary increases per year of experience; the intercept tells you where the line crosses the y-axis (theoretical salary at zero experience).

This is called **Ordinary Least Squares (OLS)** — "ordinary" because it is the standard approach, "least squares" because it minimises the sum of squared errors. The squaring is important: it penalises large errors more than small ones, and it ensures that predictions above and below the line contribute equally to the total error.

With a single feature, the model is just `Salary = slope × YearsExperience + intercept` — the same equation you learned in high school algebra, now fitted to real data. The magic is in how sklearn finds the best slope and intercept: it uses a closed-form mathematical solution (the "normal equation") that gives the exact optimal values in one step, no iteration required.

Let us load the data and see what we are working with.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"Salary_Data.csv")
print(df.shape)
print(df.head())
print(df.describe())
print(df.isnull().sum())


**What to notice:**
- 30 rows, 2 columns — a tiny, clean dataset.
- `YearsExperience` ranges from 1.1 to 10.5 years; `Salary` ranges from ~37k to ~122k.
- No missing values, so we can go straight to modeling.

> **Pitfall:** Always check for missing values before fitting. `LinearRegression` will raise an error on NaNs.

---

## 2. Fit a Linear Regression Model

Here is a question that separates careful data scientists from careless ones: how do you know your model actually works? The answer is humbling — you don't, until you test it on data it has never seen. This is why we split our data before doing anything else.

We use 70% of the data for **training** — this is the data the model learns from, the dots on our graph. We hold back 30% for **testing** — this is the exam the model takes after studying. If the model performs well on training data but poorly on test data, it has **overfit**: it memorised the training examples instead of learning the underlying pattern. The train/test split is the simplest defence against overfitting, and it is non-negotiable in any real project.

The `random_state=42` parameter ensures the split is reproducible — you and I will get the exact same train and test sets, so we can compare results fairly. Without it, every run would produce a different split, and your metrics would be noisy.

Once the data is split, fitting the model is one line: `model.fit(X_train, y_train)`. Behind the scenes, sklearn solves the normal equation to find the slope and intercept that minimise squared error on the training data. The result is a model with two key parameters: `model.coef_` (the slope — how much salary increases per year) and `model.intercept_` (the baseline salary at zero experience).

Let us fit the model and inspect what it learned.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv(r"Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Slope (coef_):     {model.coef_[0]:.2f}")
print(f"Intercept:         {model.intercept_:.2f}")
print(f"R² on train:       {model.score(X_train, y_train):.4f}")
print(f"R² on test:        {model.score(X_test, y_test):.4f}")


**What to notice:**
- The slope tells you the average salary increase per additional year of experience.
- The intercept is the model's salary prediction at 0 years (extrapolation — interpret with caution).
- R² near 0.95 means the line explains ~95% of salary variance — a strong fit.

> **Pitfall:** A high R² on training data can be misleading. Always check test R² to detect overfitting.

---

## 3. Make Predictions

A model that cannot predict is just a maths exercise. The whole point of fitting a linear regression is to use it: feed in a new value, get out a prediction. This is where the model becomes a tool rather than a curiosity.

When we call `model.predict([[5.5]])`, we are asking: "Based on the pattern you learned from 21 employees, what salary would you expect for someone with 5.5 years of experience?" The model plugs 5.5 into the equation `Salary = slope × 5.5 + intercept` and returns a number. That number is our best guess — not a guarantee, but the most defensible estimate we can make from this data.

But how good is that guess? To answer, we need a metric. **Mean Squared Error (MSE)** measures the average squared difference between predicted and actual salaries on the test set. Squaring the errors ensures that large misses are penalised more than small ones, and that over-predictions and under-predictions contribute equally. The problem with MSE is its units: if salary is in dollars, MSE is in *squared dollars* — a number that is hard to interpret. That is why we also compute **Root Mean Squared Error (RMSE)**, which takes the square root and puts the error back in dollars. An RMSE of $6,000 means the model's predictions are typically about $6,000 off — a number an HR director can actually understand.

Let us make a prediction and compute both metrics.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_csv(r"Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred_5_5 = model.predict([[5.5]])
print(f"Predicted salary at 5.5 years: ${pred_5_5[0]:,.2f}")

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Test MSE:  {mse:,.2f}")
print(f"Test RMSE: {np.sqrt(mse):,.2f}")


**What to notice:**
- The prediction at 5.5 years should be close to the actual salaries of employees with ~5 years of experience.
- RMSE is in the same units as salary (dollars), making it more interpretable than MSE.

> **Pitfall:** `model.predict` expects a 2D array. Use `[[5.5]]`, not `[5.5]`.

---

## 4. Residual Analysis

Here is a secret that separates junior analysts from senior ones: the model's coefficients tell you what the model learned, but the **residuals** tell you whether to trust it. A residual is simply the difference between the actual value and the predicted value: `residual = actual - predicted`. If the model is good, residuals should look like random noise — scattered around zero with no discernible pattern. If they show a pattern, the model is missing something.

Plotting residuals against predicted values is the single most important diagnostic in regression analysis. Here is what to look for:

- **Random scatter around zero**: The model has captured the underlying pattern. The remaining error is irreducible noise.
- **Funnel shape (heteroscedasticity)**: The spread of residuals changes with the prediction. This means the model is more confident in some regions than others — a sign that a transformation (like log-scaling the target) might help.
- **Curved pattern**: The relationship is not linear. A straight line is the wrong shape for this data, and you need polynomial features or a different model.
- **Outliers**: A few points with huge residuals may be data errors or special cases that deserve investigation.

The mean residual should be near zero (the model is unbiased on average), and the standard deviation tells you the typical error size. Let us plot the residuals and see what story they tell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv(r"Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_pred, residuals, color="steelblue", edgecolor="k", alpha=0.7)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Predicted Salary")
ax.set_ylabel("Residual (Actual − Predicted)")
ax.set_title("Residual Plot — Salary Prediction")
plt.tight_layout()
from IPython.display import display
display(plt.gcf())

print(f"Mean residual: {residuals.mean():.2f}")
print(f"Residual std:  {residuals.std():.2f}")


**What to notice:**
- Residuals should be roughly symmetric around the red zero line.
- A mean near zero confirms the model is unbiased on average.
- No funnel shape or curve means the linear assumption is reasonable.

> **Pitfall:** A funnel-shaped residual plot (heteroscedasticity) suggests the error variance changes with the prediction — a sign the model may need transformation.

---

## Exercises (do these!)

### Exercise 1 — Predict Salary at 5.5 Years
Use the fitted model to predict the salary for a candidate with 5.5 years of experience.
*Expected: $77,283.39*

**Follow-up:** What is the predicted salary at 0 years? Why might this be unrealistic? Check: intercept value.

### Exercise 2 — Compute Test MSE
Calculate the Mean Squared Error on the held-out test set.
*Expected: MSE = 37,784,662.47 (RMSE = $6,146.92)*

**Follow-up:** Why is RMSE more interpretable than MSE? Check: RMSE value.

### Exercise 3 — Interpret R²
The test R² is approximately 0.95. What does this mean in plain language?
*Expected: The model explains 94.1% of the variance in salary using years of experience.*

**Follow-up:** What would R² = 0 mean? What would R² = 1 mean? Check: R² on train vs test.

<details>
<summary>Hint</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv(r"Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# Exercise 1
pred = model.predict([[5.5]])
print(pred[0])

# Exercise 2
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, model.predict(X_test))
print(mse)

# Exercise 3
print(model.score(X_test, y_test))
```

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_csv(r"Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# --- Solution 1 ---
pred_5_5 = model.predict([[5.5]])
print(f"Predicted salary at 5.5 years: ${pred_5_5[0]:,.2f}")

# --- Solution 2 ---
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Test MSE:  {mse:,.2f}")
print(f"Test RMSE: {np.sqrt(mse):,.2f}")

# --- Solution 3 ---
r2_train = model.score(X_train, y_train)
r2_test = model.score(X_test, y_test)
print(f"R² train: {r2_train:.4f}")
print(f"R² test:  {r2_test:.4f}")
print(f"The model explains {r2_test*100:.1f}% of salary variance.")

# --- Follow-up 1 ---
print(f"Intercept (salary at 0 years): ${model.intercept_:,.2f}")

# --- Follow-up 2 ---
print(f"RMSE: ${np.sqrt(mse):,.2f} — in salary dollars, unlike MSE.")

# --- Follow-up 3 ---
assert r2_test > 0.90, "R² should be above 0.90"
print(f"R² test = {r2_test:.4f} — strong fit confirmed.")


### What to learn next
- Multiple linear regression with more features
- Regularization (Ridge, Lasso) to prevent overfitting
- Polynomial regression for non-linear relationships

*Files in this folder: Salary_Data.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
